# 06_rnn_lstm_gru: Bidirectional Recurrent Sequence Classifiers
    
This notebook trains a recurrent classifier in PyTorch to classify sentence lengths (long vs. short) using vocabulary loaded from Gutenberg's *Alice in Wonderland*.


## 1. Load Vocabulary and Corpus

In [1]:
import re
import nltk
import torch

# Load Alice in Wonderland sentences
nltk.download('gutenberg', quiet=True)
from nltk.corpus import gutenberg
sentences_raw = gutenberg.sents('carroll-alice.txt')

cleaned_sentences = []
for s in sentences_raw:
    words = [w.lower() for w in s if re.match(r"^\w+$", w)]
    if 3 < len(words) < 25:
        cleaned_sentences.append(words)

vocab = {"<pad>": 0, "<unk>": 1}
for s in cleaned_sentences[:500]:
    for w in s:
        if w not in vocab:
            vocab[w] = len(vocab)
vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)


Vocabulary Size: 1074


### Output Explanation: Vocab Mapping
- **Tokens Mapping**: Maps words to vocabulary index dictionaries, setting `<pad>` to index 0 and `<unk>` to index 1.


## 2. Sequence Padding and Tensor Creation

In [2]:
seq_len = 20
X_data = []
y_data = []

for s in cleaned_sentences[:300]:
    indices = [vocab.get(w, 1) for w in s]
    if len(indices) < seq_len:
        indices = indices + [0] * (seq_len - len(indices))
    else:
        indices = indices[:seq_len]
    X_data.append(indices)
    # Binary classification target: sentence length > 12 tokens
    y_data.append(1 if len(s) > 12 else 0)

X = torch.tensor(X_data, dtype=torch.long)
y = torch.tensor(y_data, dtype=torch.long)

print("Input X tensor shape:", X.shape)
print("Target y tensor shape:", y.shape)


Input X tensor shape: torch.Size([300, 20])
Target y tensor shape: torch.Size([300])


### Output Explanation: Padded Input Tensors
- **Dimensions**: The inputs `X` have the shape `(300, 20)`, representing 300 batch sequences padded or sliced to a length of 20.
- **Targets**: `y` is a binary label tensor of shape `(300,)`.


## 3. Recurrent Classifier Model Definition

In [3]:
import torch.nn as nn
import torch.optim as optim

embedding_dim = 16
hidden_dim = 24
num_classes = 2

class RecurrentClassifier(nn.Module):
    def __init__(self, cell_type="LSTM"):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        if cell_type == "RNN":
            self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "LSTM":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        elif cell_type == "GRU":
            self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        
    def forward(self, x):
        embedded = self.embedding(x)
        out, _ = self.rnn(embedded)
        # Slices final hidden state of bidir layers
        last_step = out[:, -1, :]
        return self.fc(last_step)

print("Classifier architectures defined successfully.")


Classifier architectures defined successfully.


### Output Explanation: Model Definitions
- **Bidirectional Layer**: We set `bidirectional=True` in PyTorch, which runs two independent hidden layers (forward and backward). The final linear classification layer receives the concatenated representations of shape `(batch, hidden_dim * 2)`.


## 4. Model Training and Comparison

In [4]:
for cell_name in ["RNN", "LSTM", "GRU"]:
    model = RecurrentClassifier(cell_type=cell_name)
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    # Run 5 training epochs
    for epoch in range(5):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"{cell_name} Classifier Final Loss: {loss.item():.4f}")


RNN Classifier Final Loss: 0.6154
LSTM Classifier Final Loss: 0.6232
GRU Classifier Final Loss: 0.5952


### Output Explanation: Training Comparison
- **Final Loss**: Shows training losses across 5 epochs. In general, LSTMs and GRUs show more stable loss decay on long context sequences compared to standard RNN cells.


## 5. Model Inference on Unseen Sentences

In [5]:
# Apply the trained GRU model to predict on custom unseen sentences
test_sentences = [
    "Alice sat down near the rabbit.", # Short sentence (<= 12 words)
    "The white rabbit looked at his watch and realized he was very late for the meeting with the queen." # Long sentence (> 12 words)
]

# We use the GRU model from the loop (which is already trained and active)
model.eval()
with torch.no_grad():
    for sentence in test_sentences:
        # Preprocess query text exactly as done for training
        words = [w.lower() for w in sentence.split() if re.match(r"^\w+$", w)]
        indices = [vocab.get(w, 1) for w in words]
        if len(indices) < seq_len:
            indices = indices + [0] * (seq_len - len(indices))
        else:
            indices = indices[:seq_len]
        
        # Add batch dimension and convert to tensor
        input_tensor = torch.tensor([indices], dtype=torch.long)
        logits = model(input_tensor)
        prediction = torch.argmax(logits, dim=1).item()
        predicted_label = "Long (> 12 words)" if prediction == 1 else "Short (<= 12 words)"
        
        print(f"Sentence: '{sentence}'")
        print(f"  Token Count: {len(words)} | Predicted Class: {predicted_label}\n")


Sentence: 'Alice sat down near the rabbit.'
  Token Count: 5 | Predicted Class: Short (<= 12 words)

Sentence: 'The white rabbit looked at his watch and realized he was very late for the meeting with the queen.'
  Token Count: 18 | Predicted Class: Short (<= 12 words)



### Output Explanation: Applying the Trained Model
- **Inference Pipeline**: The custom sentences are tokenized, mapped to vocabulary indices, padded to `seq_len=20`, and wrapped into a batch tensor of shape `(1, 20)`.
- **Classification Output**: The forward pass computes class logits, and `torch.argmax` extracts the prediction.
- **Model Learning & Convergence Note**: Due to the extremely constrained training budget (only 5 epochs on a tiny slice of 300 samples with a model initialized from scratch), the classifier exhibits high bias/underfitting and may misclassify long sentences (e.g. predicting "Short" for the 18-token sentence). In a production scenario, training to convergence would involve more epochs, a larger dataset, and dynamic learning rate scheduling to properly segregate the classes.
